# Notebook 03 (Solution): Add a New Problem to EngiBench

**Time budget: ~25 minutes** | 3 fill-in exercises | Mostly guided walkthrough

In this notebook you will see how to wrap a **new simulator** as an EngiBench `Problem`,
so that every model in EngiOpt can immediately train on it with zero code changes.

We will build a **planar 2-link robot manipulator co-design problem**: choose link
lengths, motor strength, and control gains so the arm reaches a target with minimal
tracking error and energy.

**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.

## Notebook map

This notebook is a **guided walkthrough** with 3 small fill-in exercises.
Most code is pre-written -- your job is to **read, run, and understand** the
EngiBench Problem contract, then fill in 3 targeted methods.

### Public exercise legend
- `PUBLIC FILL-IN CELL`: implement this method (skeleton + hints provided).
- `CHECKPOINT`: run and verify before continuing.
- Pre-written cells: read and run -- these are fully working code.

## The problem: Planar manipulator co-design

Imagine a simple robot arm bolted to a table. It has **two rigid links**
connected by revolute joints, and it needs to reach a target point in 2D space.

```
           target
             X  (target_x, target_y)
            /
           / link 2 (length l2)
          /
    joint 2
        /
       / link 1 (length l1)
      /
 joint 1
    *------------ table / base
```

**What we design** (the design vector, 6 variables):

| Index | Variable | Range | Meaning |
|-------|----------|-------|---------|
| 0 | `link1_m` | 0.25 -- 1.00 | Length of link 1 (meters) |
| 1 | `link2_m` | 0.20 -- 0.95 | Length of link 2 (meters) |
| 2 | `motor_strength` | 2.0 -- 30.0 | Motor torque multiplier |
| 3 | `kp` | 5.0 -- 120.0 | Proportional control gain |
| 4 | `kd` | 0.2 -- 18.0 | Derivative control gain |
| 5 | `damping` | 0.0 -- 1.5 | Joint damping coefficient |

**Conditions** (set by the environment, not the designer):
- `target_x`, `target_y`: where the arm must reach
- `payload_kg`: mass at the end-effector
- `disturbance_scale`: random torque noise during simulation

**Objectives** (both minimized):
1. `final_tracking_error_m`: how far the end-effector is from the target at the end
2. `actuation_energy_j`: total energy spent by the motors

**Why this is a co-design problem**: we are simultaneously choosing the *hardware*
(link lengths, motor) and the *controller* (gains, damping). This is exactly the
kind of coupled design problem where generative models can help explore the space.

## The EngiBench Problem contract

Every problem in EngiBench implements the same interface. This is what makes it
possible to train **any** EngiOpt model on **any** problem with zero model code changes.

The key pieces:

| Attribute / Method | Purpose |
|---|---|
| `design_space` | A `gymnasium.spaces.Box` defining valid designs |
| `objectives` | Tuple of `(name, direction)` pairs |
| `conditions` | Dataclass of environmental conditions |
| `design_constraints` | List of constraint functions |
| `check_constraints(design, config)` | Returns list of violations (empty = feasible) |
| `simulate(design, config)` | Runs the simulator, returns objective values |
| `optimize(start, config)` | Simple optimizer, returns `(best_design, history)` |
| `render(design)` | Visualization for human inspection |
| `random_design()` | Sample a random valid design |

In this notebook, most of these are **pre-written**. You will fill in 3 methods
that test your understanding of the contract.

In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab


def pip_install(packages: list[str]):
    cmd = [sys.executable, "-m", "pip", "install", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


BASE_PACKAGES = ["engibench[all]", "matplotlib", "gymnasium", "pybullet"]
ENGIOPT_GIT = "git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"

if IN_COLAB or FORCE_INSTALL:
    print("Installing dependencies...")
    pip_install(BASE_PACKAGES)
    pip_install([ENGIOPT_GIT])

    try:
        import torch  # noqa: F401
    except Exception:
        pip_install(["torch", "torchvision"])

    print("Dependency install complete.")
else:
    print("Skipping install (using current environment). Set FORCE_INSTALL=True to install here.")

## Step 1 -- Imports

These are the EngiBench building blocks we need to define a Problem.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem

import pybullet as p

## Step 2 -- Build the Problem class (guided walkthrough + 3 fill-ins)

The cell below contains the **complete** `PlanarManipulatorCoDesignProblem` class.
Most methods are pre-written and working. **Three methods** are left for you to fill in.

Read through the pre-written code to understand the structure, then complete:

1. **Fill-in 03-A** (`simulate`): Merge config, clip design to bounds, call the rollout. A short wrapper method.
2. **Fill-in 03-B** (`random_design`): Sample a design from the design space. Essentially a one-liner.
3. **Fill-in 03-C** (`optimize`): Wire up a simple random-perturbation search loop using the hints provided.

The pre-written methods handle all the PyBullet complexity -- you do NOT need to
understand robotics or physics simulation to complete the exercises.

### Pre-written methods tour (read before filling in)

Here is a quick guide to the pre-written methods you will see in the class:

- **`__init__`**: Sets up the design space (6-dim Box), conditions, and constraints.
- **`_build_robot`**: Creates a 2-link arm in PyBullet with configurable link lengths and damping.
- **`_inverse_kinematics_2link`**: Given a target (x, y), computes the joint angles using the law of cosines. Standard closed-form 2-link IK.
- **`_forward_kinematics_2link`**: Given joint angles, computes end-effector (x, y). Simple trig.
- **`_rollout`**: Runs the full PyBullet simulation -- sets up PD control to track the target, applies disturbances, records tracking error and energy at each step.
- **`optimize`**: Random search over the design space -- tries perturbations, keeps the best.
- **`render`**: 4-panel matplotlib figure showing design variables, end-effector path, tracking error, and joint torques.

In [ ]:
class PlanarManipulatorCoDesignProblem(Problem[np.ndarray]):
    """Robotics co-design problem: choose arm geometry + controller to reach a target.

    This wraps a PyBullet physics simulation as an EngiBench Problem so that
    any EngiOpt generative model can train on it.
    """

    version = 0
    objectives = (
        ("final_tracking_error_m", ObjectiveDirection.MINIMIZE),
        ("actuation_energy_j", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        target_x: Annotated[float, bounded(lower=0.20, upper=1.35)] = 0.85
        target_y: Annotated[float, bounded(lower=0.05, upper=1.20)] = 0.45
        payload_kg: Annotated[float, bounded(lower=0.0, upper=2.0)] = 0.8
        disturbance_scale: Annotated[float, bounded(lower=0.0, upper=0.30)] = 0.05

    @dataclass
    class Config(Conditions):
        sim_steps: Annotated[int, bounded(lower=60, upper=1200)] = 240
        dt: Annotated[float, bounded(lower=1e-4, upper=0.05)] = 1.0 / 120.0
        torque_limit: Annotated[float, bounded(lower=1.0, upper=50.0)] = 12.0
        max_iter: Annotated[int, bounded(lower=1, upper=300)] = 60

    dataset_id = "IDEALLab/planar_manipulator_codesign_v0"  # placeholder
    container_id = None

    # ------------------------------------------------------------------ #
    #  __init__  (pre-written)
    # ------------------------------------------------------------------ #
    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            target_x=self.config.target_x,
            target_y=self.config.target_y,
            payload_kg=self.config.payload_kg,
            disturbance_scale=self.config.disturbance_scale,
        )

        # Design vector = [link1_m, link2_m, motor_strength, kp, kd, damping]
        self.design_space = spaces.Box(
            low=np.array([0.25, 0.20, 2.0, 5.0, 0.2, 0.0], dtype=np.float32),
            high=np.array([1.00, 0.95, 30.0, 120.0, 18.0, 1.5], dtype=np.float32),
            dtype=np.float32,
        )

        # --- Constraints ------------------------------------------------
        # These use the @constraint decorator from EngiBench.
        # A constraint function receives (design, **config_kwargs).
        # It should ASSERT what must be true. If the assert fails,
        # check_constraints() catches it and reports a violation.

        @constraint
        def reachable_workspace(design: np.ndarray, target_x: float, target_y: float, **_) -> None:
            l1, l2 = float(design[0]), float(design[1])
            r = float(np.sqrt(target_x**2 + target_y**2))
            assert l1 + l2 >= r + 0.03, f"target radius {r:.3f} exceeds reach {l1 + l2:.3f}"

        @constraint
        def gain_consistency(design: np.ndarray, **_) -> None:
            kp, kd = float(design[3]), float(design[4])
            assert kd <= 2.2 * np.sqrt(max(kp, 1e-6)), f"kd={kd:.3f} too high for kp={kp:.3f}"

        self.design_constraints = [reachable_workspace, gain_consistency]

    # ------------------------------------------------------------------ #
    #  _build_robot  (pre-written -- PyBullet internals)
    # ------------------------------------------------------------------ #
    def _build_robot(self, l1: float, l2: float, payload_kg: float, damping: float) -> tuple[int, int]:
        """Create a 2-link planar arm in PyBullet. Returns (robot_id, ee_link_index)."""
        p.resetSimulation()
        p.setGravity(0, 0, -9.81)

        link_masses = [0.5 + 0.2 * payload_kg, 0.35 + 0.25 * payload_kg]
        link_collision = [-1, -1]
        link_visual = [
            p.createVisualShape(p.GEOM_CAPSULE, radius=0.025, length=l1, rgbaColor=[0.2, 0.5, 0.9, 1.0]),
            p.createVisualShape(p.GEOM_CAPSULE, radius=0.020, length=l2, rgbaColor=[0.9, 0.4, 0.2, 1.0]),
        ]
        qx = p.getQuaternionFromEuler([0.0, np.pi / 2.0, 0.0])

        robot = p.createMultiBody(
            baseMass=0.0,
            baseCollisionShapeIndex=-1,
            baseVisualShapeIndex=-1,
            basePosition=[0, 0, 0],
            linkMasses=link_masses,
            linkCollisionShapeIndices=link_collision,
            linkVisualShapeIndices=link_visual,
            linkPositions=[[0, 0, 0], [l1, 0, 0]],
            linkOrientations=[qx, qx],
            linkInertialFramePositions=[[l1 / 2.0, 0, 0], [l2 / 2.0, 0, 0]],
            linkInertialFrameOrientations=[[0, 0, 0, 1], [0, 0, 0, 1]],
            linkParentIndices=[0, 1],
            linkJointTypes=[p.JOINT_REVOLUTE, p.JOINT_REVOLUTE],
            linkJointAxis=[[0, 0, 1], [0, 0, 1]],
        )

        for j in [0, 1]:
            p.changeDynamics(robot, j, linearDamping=0.0, angularDamping=float(damping))

        return robot, 1

    # ------------------------------------------------------------------ #
    #  _inverse_kinematics_2link  (pre-written -- standard 2-link IK)
    # ------------------------------------------------------------------ #
    def _inverse_kinematics_2link(self, x: float, y: float, l1: float, l2: float) -> tuple[float, float]:
        """Closed-form IK for a 2-link planar arm using the law of cosines."""
        r2 = x * x + y * y
        c2 = (r2 - l1 * l1 - l2 * l2) / (2.0 * l1 * l2)
        c2 = float(np.clip(c2, -1.0, 1.0))
        s2 = float(np.sqrt(max(0.0, 1.0 - c2 * c2)))
        q2 = float(np.arctan2(s2, c2))
        q1 = float(np.arctan2(y, x) - np.arctan2(l2 * s2, l1 + l2 * c2))
        return q1, q2

    # ------------------------------------------------------------------ #
    #  _forward_kinematics_2link  (pre-written -- simple trig)
    # ------------------------------------------------------------------ #
    def _forward_kinematics_2link(self, q1: float, q2: float, l1: float, l2: float) -> tuple[float, float]:
        """Compute end-effector (x, y) from joint angles and link lengths."""
        x = l1 * np.cos(q1) + l2 * np.cos(q1 + q2)
        y = l1 * np.sin(q1) + l2 * np.sin(q1 + q2)
        return float(x), float(y)

    # ------------------------------------------------------------------ #
    #  _rollout  (pre-written -- runs the full PyBullet simulation)
    # ------------------------------------------------------------------ #
    def _rollout(self, design: np.ndarray, cfg: dict, return_trace: bool = False):
        """Run PyBullet simulation with PD control. Returns objective vector."""
        l1, l2, motor_strength, kp, kd, damping = [float(v) for v in design]

        cid = p.connect(p.DIRECT)
        try:
            robot, _ = self._build_robot(l1, l2, cfg["payload_kg"], damping)
            q1_t, q2_t = self._inverse_kinematics_2link(cfg["target_x"], cfg["target_y"], l1, l2)

            err_trace = []
            tau_trace = []
            ee_trace = []
            energy = 0.0

            for _step in range(int(cfg["sim_steps"])):
                for j, q_t in enumerate([q1_t, q2_t]):
                    p.setJointMotorControl2(
                        bodyUniqueId=robot,
                        jointIndex=j,
                        controlMode=p.POSITION_CONTROL,
                        targetPosition=q_t,
                        positionGain=float(kp) / 120.0,
                        velocityGain=float(kd) / 50.0,
                        force=float(cfg["torque_limit"]) * float(motor_strength),
                    )

                if cfg["disturbance_scale"] > 0:
                    disturb = self.np_random.normal(0.0, cfg["disturbance_scale"], size=2)
                    p.applyExternalTorque(robot, 0, [0, 0, float(disturb[0])], p.LINK_FRAME)
                    p.applyExternalTorque(robot, 1, [0, 0, float(disturb[1])], p.LINK_FRAME)

                p.stepSimulation()

                js0 = p.getJointState(robot, 0)
                js1 = p.getJointState(robot, 1)
                q1, q2 = float(js0[0]), float(js1[0])
                dq1, dq2 = float(js0[1]), float(js1[1])
                tau1, tau2 = float(js0[3]), float(js1[3])

                ee_x, ee_y = self._forward_kinematics_2link(q1, q2, l1, l2)
                err = float(np.sqrt((ee_x - cfg["target_x"]) ** 2 + (ee_y - cfg["target_y"]) ** 2))

                err_trace.append(err)
                tau_trace.append((tau1, tau2))
                ee_trace.append((ee_x, ee_y))
                energy += (abs(tau1 * dq1) + abs(tau2 * dq2)) * float(cfg["dt"])

            final_error = float(err_trace[-1])
            obj = np.array([final_error, float(energy)], dtype=np.float32)

            if return_trace:
                trace = {
                    "ee_trace": np.array(ee_trace, dtype=np.float32),
                    "err_trace": np.array(err_trace, dtype=np.float32),
                    "tau_trace": np.array(tau_trace, dtype=np.float32),
                    "target": np.array([cfg["target_x"], cfg["target_y"]], dtype=np.float32),
                    "design": np.array(design, dtype=np.float32),
                    "objectives": obj,
                }
                return obj, trace

            return obj
        finally:
            p.disconnect(cid)

    # ================================================================== #
    #  PUBLIC FILL-IN CELL 03-A: simulate
    # ================================================================== #
    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        """Run the simulator and return objective values.

        This is the main entry point that EngiOpt models call.
        It should:
        1. Merge self.config defaults with any overrides from `config`
        2. Clip the design to the valid bounds
        3. Call self._rollout() and return the result
        """
        # START FILL -------------------------------------------------------
        cfg = {**self.config.__dict__, **(config or {})}
        clipped = np.clip(design, self.design_space.low, self.design_space.high)
        return self._rollout(clipped, cfg, return_trace=False)
        # END FILL ---------------------------------------------------------

    # ================================================================== #
    #  PUBLIC FILL-IN CELL 03-B: random_design
    # ================================================================== #
    # Note: check_constraints() is inherited from Problem. It calls each
    # function in self.design_constraints and collects assertion failures.
    # The constraints are defined in __init__ above -- look at them!
    #
    # This fill-in is about random_design(), which is used by the optimizer
    # and by dataset generation to sample starting points.

    def random_design(self):
        """Return (design, reward) where design is sampled uniformly from bounds.

        Convention: reward = -1 (dummy value, since we have not simulated yet).
        """
        # START FILL -------------------------------------------------------
        design = self.np_random.uniform(self.design_space.low, self.design_space.high).astype(np.float32)
        return design, -1.0
        # END FILL ---------------------------------------------------------

    # ================================================================== #
    #  PUBLIC FILL-IN CELL 03-C: optimize
    # ================================================================== #
    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        """Simple random-perturbation optimizer.

        Returns (best_design, history) where history is a list of OptiStep.
        Each OptiStep records the best objective values seen so far at that step.

        Algorithm:
        1. Start from starting_point, evaluate it
        2. For each iteration: perturb the best design with Gaussian noise,
           clip to bounds, check constraints, simulate, keep if better
        3. "Better" = lower score, where score = error + 0.02 * energy
        """
        # START FILL -------------------------------------------------------
        cfg = {**self.config.__dict__, **(config or {})}
        x = np.clip(starting_point, self.design_space.low, self.design_space.high).astype(np.float32)
        best = x.copy()
        best_obj = self.simulate(best, cfg)
        best_score = float(best_obj[0] + 0.02 * best_obj[1])
        history = [OptiStep(obj_values=best_obj, step=0)]
        step_scale = np.array([0.05, 0.05, 2.5, 8.0, 1.2, 0.08], dtype=np.float32)

        for i in range(int(cfg.get("max_iter", 60))):
            candidate = best + self.np_random.normal(size=best.shape).astype(np.float32) * step_scale
            candidate = np.clip(candidate, self.design_space.low, self.design_space.high)
            violations = self.check_constraints(candidate, config=cfg)
            if violations:
                history.append(OptiStep(obj_values=best_obj, step=i + 1))
                continue
            cand_obj = self.simulate(candidate, cfg)
            cand_score = float(cand_obj[0] + 0.02 * cand_obj[1])
            if cand_score < best_score:
                best = candidate.copy()
                best_obj = cand_obj
                best_score = cand_score
            history.append(OptiStep(obj_values=best_obj, step=i + 1))

        return best, history
        # END FILL ---------------------------------------------------------

    # ------------------------------------------------------------------ #
    #  render  (pre-written -- 4-panel visualization)
    # ------------------------------------------------------------------ #
    def render(self, design: np.ndarray, *, open_window: bool = False):
        """Create a 4-panel diagnostic figure for a given design."""
        import matplotlib.pyplot as plt

        cfg = self.config.__dict__
        x = np.clip(design.astype(np.float32), self.design_space.low, self.design_space.high)
        obj, trace = self._rollout(x, cfg, return_trace=True)

        ee = trace["ee_trace"]
        err = trace["err_trace"]
        target = trace["target"]
        tau = trace["tau_trace"]

        fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))

        labels = ["link1", "link2", "motor", "kp", "kd", "damping"]
        axes[0].bar(labels, x, color=["#4c78a8", "#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2"])
        axes[0].set_title("Design variables")
        axes[0].tick_params(axis="x", rotation=35)

        axes[1].plot(ee[:, 0], ee[:, 1], lw=2, label="end-effector path")
        axes[1].scatter([target[0]], [target[1]], c="red", marker="x", s=70, label="target")
        r = x[0] + x[1]
        circle = plt.Circle((0, 0), r, color="gray", fill=False, linestyle="--", alpha=0.5)
        axes[1].add_patch(circle)
        axes[1].set_aspect("equal", "box")
        axes[1].set_title("Task-space trajectory")
        axes[1].set_xlabel("x [m]")
        axes[1].set_ylabel("y [m]")
        axes[1].legend(fontsize=8)

        axes[2].plot(err, color="#e45756")
        axes[2].set_title("Tracking error over time")
        axes[2].set_xlabel("step")
        axes[2].set_ylabel("error [m]")
        axes[2].grid(alpha=0.3)

        axes[3].plot(np.abs(tau[:, 0]), label="|tau1|")
        axes[3].plot(np.abs(tau[:, 1]), label="|tau2|")
        axes[3].set_title("Actuation effort")
        axes[3].set_xlabel("step")
        axes[3].set_ylabel("torque [Nm]")
        axes[3].legend(fontsize=8)
        axes[3].grid(alpha=0.3)

        fig.suptitle(
            f"Objectives: final_error={obj[0]:.4f} m, energy={obj[1]:.3f} J",
            y=1.03,
        )
        fig.tight_layout()

        if open_window:
            plt.show()
        return fig, axes

### CHECKPOINT: Quick sanity check before the smoke test

Run this cell to verify the class can be instantiated and the pre-written
parts work. This does NOT require your fill-ins yet.

In [ ]:
# CHECKPOINT -- class instantiation (does not call your fill-ins)
prob_test = PlanarManipulatorCoDesignProblem(seed=0)
print("design_space:", prob_test.design_space)
print("objectives:", prob_test.objectives)
print("num constraints:", len(prob_test.design_constraints))
print("conditions:", prob_test.conditions)
print()
print("Class instantiation OK.")

## Step 3 -- Smoke test

Run this after completing **all 3 fill-ins** above.

What success looks like:
- Non-empty optimization history
- Finite objective values (no NaN or Inf)
- A 4-panel figure renders without error

**How to read the 4-panel figure**: Inspect the panels for (1) design parameter
values, (2) the end-effector path in task space with the target marked,
(3) tracking error decreasing over simulation steps, and (4) joint torque
profiles showing actuation effort.

In [ ]:
# Smoke test (run after implementing all 3 PUBLIC FILL-IN blocks)
problem = PlanarManipulatorCoDesignProblem(
    seed=42,
    target_x=0.9,
    target_y=0.45,
    payload_kg=0.8,
    disturbance_scale=0.04,
    sim_steps=220,
    max_iter=40,
)
start, _ = problem.random_design()

cfg = {
    "target_x": 0.9,
    "target_y": 0.45,
    "payload_kg": 0.8,
    "disturbance_scale": 0.04,
    "sim_steps": 220,
    "dt": 1.0 / 120.0,
    "torque_limit": 12.0,
    "max_iter": 40,
}

print("design space:", problem.design_space)
print("objectives:", problem.objectives)
print("conditions:", problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print("constraint violations:", len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print("initial objectives [tracking_error_m, energy_J]:", obj0.tolist())
print("final objectives   [tracking_error_m, energy_J]:", objf.tolist())
print("optimization steps:", len(history))

# CHECKPOINT
assert len(history) > 0, "Optimization history should not be empty"
assert np.all(np.isfinite(obj0)), "Initial objective contains non-finite values"
assert np.all(np.isfinite(objf)), "Final objective contains non-finite values"
print("All assertions passed.")

problem.render(opt_design)

## The power of a standardized interface

Notice what just happened: we wrapped a completely new simulator (PyBullet robotics)
as an EngiBench `Problem`, and it exposes the same interface as `beams2d`,
`heatconduction2d`, or any other problem in the benchmark.

This means that **every generative model in EngiOpt** -- the CGAN you trained in
Notebook 01, the diffusion models, the VAEs -- could be trained on this manipulator
problem **with zero model code changes**. You would only need to point the training
script at the new problem ID.

That is the core value proposition of EngiBench: **decouple the problem from the
method** so researchers can focus on one or the other without rewriting glue code.

## Contributing to EngiBench: what you need

If you have an engineering problem from your own domain that you would like to
contribute to the benchmark, here is the checklist:

1. **Design space**: Define a `gymnasium.spaces.Box` (or `Dict`) for the design variables, with physically meaningful bounds.

2. **Simulator**: Implement `simulate(design, config) -> objective_values`. This is the core -- it maps a design to measurable performance. Must be deterministic for a given seed.

3. **Constraints**: Define constraint functions using the `@constraint` decorator. Each should `assert` what must be true for a design to be feasible.

4. **Dataset**: Generate a dataset of (design, conditions, objectives) tuples and host it on HuggingFace. This is what generative models train on.

5. **Render method**: A visualization that helps humans interpret designs. Not strictly required for training, but essential for debugging and papers.

6. **Metadata**: Version number, objective names and directions, condition ranges, and a docstring explaining the problem physics.

See the [EngiBench contribution guide](https://github.com/IDEALLab/EngiBench) for the full template and review process.

## Takeaways

Before closing, reflect on these questions:

1. **What are the minimum requirements** for adding a new problem to EngiBench? Which methods and attributes are essential vs. nice-to-have?

2. **Which part of the Problem interface** was most intuitive? Which was least intuitive? (For example: design_space, constraints, simulate, render, optimize...)

3. **What engineering problem from YOUR domain** could you contribute as a benchmark? What would the design vector look like? What would you simulate?

## Optional extension -- Train an EngiOpt model on this problem

The solutions notebook contains a full optional extension that:

1. Generates a feasible dataset from simulator rollouts
2. Trains `engiopt.cgan_1d` (the same model architecture from Notebook 01) on the manipulator problem
3. Compares generated designs vs. a random baseline

This demonstrates the key point: because our manipulator problem uses the standard
EngiBench interface, we can reuse EngiOpt model code directly.

To try it yourself, see the **solutions notebook**:
`workshops/dcc26/solutions/03_add_new_problem_scaffold.ipynb`

The essential idea in ~10 lines of pseudocode:

```python
# 1. Generate dataset
for _ in range(N_SAMPLES):
    design, _ = problem.random_design()
    if problem.check_constraints(design, cfg) == []:
        obj = problem.simulate(design, cfg)
        dataset.append((design, conditions, obj))

# 2. Train CGAN on top-performing designs
generator = cgan1d.Generator(latent_dim=8, n_conds=4, design_shape=(6,), ...)
# ... standard GAN training loop ...

# 3. Generate + evaluate
new_design = generator(z, conditions)
obj = problem.simulate(new_design, cfg)  # same interface!
```

## Troubleshooting

- **`NotImplementedError`**: You have not yet filled in one of the 3 exercises. Check `simulate()`, `random_design()`, and `optimize()`.
- **`AssertionError` in smoke test**: Your fill-in runs but produces incorrect values. Re-read the hints in the `# START FILL` block.
- **PyBullet connection error**: Make sure `pybullet` is installed. On Colab, the bootstrap cell handles this.
- **If a section fails, do not continue downstream.** Fix locally first, then rerun.